<br>
<br>
<br>
<div dir="ltr" align="center">

  <span style="font-size: 60px; color: #0B3C78; font-weight: 700;">
    Artificial Intelligence
  </span><br><br>

  <span style="font-size: 22px; color: #1F5FA8; font-weight: 500;">
    Computer Engineering Department
  </span><br>

  <span style="font-size: 20px; color: #1F5FA8; font-weight: 500;">
    Sharif University of Technology
  </span><br><br>

  <span style="font-size: 22px; color: #1F7A8C; font-weight: 600;">
    Spring 2026
  </span><br><br>

  <span style="font-size: 24px; color: #145DA0; font-weight: 700;">
    Practical Assignment
  </span><br><br>

  <span style="font-size: 22px; color: #3A7CC2; font-weight: 600;">
    Bayesian Networks
  </span><br><br><br>

</div>

---

- Instructor: Dr.Tanghatari
- Practical Designer: Iliya Forsati

---

### Student Information

- Name: Arvin
- Last Name: Baghal Asl
- Student Number: 403105793

---

### Instructions

- Make sure to:
  - Fill in your personal information above.
  - Complete all the `TODO` parts in the `BayesianNetworks.py` file.
  - Run all the cells in the notebook and make sure your code runs correctly.

## CONTENTS

- Bayesian Networks (100 points + 20 bonus)
    - BayesNode (10 points)
    - BayesNet (10 points)
    - Exact Inference in Bayesian Networks
        - Enumeration (20 points)
        - Variable elimination (20 points)
    - Approximate Inference in Bayesian Networks
        - Prior sample (10 points)
        - Rejection sampling (10 points)
        - Likelihood weighting (20 points)
        - Gibbs sampling (20 points)

In [1]:
from BayesianNetworks import *

# Bayesian Networks

A Bayesian network is a probabilistic graphical model that represents a set of 
variables and their conditional dependencies via a directed acyclic graph (DAG).

In this implementation, a Bayesian network is built using two main classes:
- **BayesNet**: The overall network structure.
- **BayesNode**: Individual nodes in the network, each representing a random variable.

This implementation focuses only on boolean variables. Each node contains a 
**conditional probability table (CPT)** which represents the probability distribution 
of that variable given its parents: **P(X | parents)**.

Let us dive into the **BayesNode** implementation.

In [2]:
psource(BayesNode)

class BayesNode:
    """A conditional probability distribution for a boolean variable,
    P(X | parents). Part of a BayesNet."""

    def __init__(self, X, parents, cpt):
        """X is a variable name, and parents a sequence of variable
        names or a space-separated string. cpt, the conditional
        probability table, takes one of these forms:

        * A number, the unconditional probability P(X=true). You can
          use this form when there are no parents.

        * A dict {v: p, ...}, the conditional probability distribution
          P(X=true | parent=v) = p. When there's just one parent.

        * A dict {(v1, v2, ...): p, ...}, the distribution P(X=true |
          parent1=v1, parent2=v2, ...) = p. Each key must have as many
          values as there are parents. You can use this form always;
          the first two are just conveniences.

        In all cases the probability of X being false is left implicit,
        since it follows from P(X=true).

        >>>

The constructor takes the name of the **variable**, **parents**, and **cpt**. 
Here **variable** is the name of the variable like 'Earthquake'. 
**parents** should be a list or space-separated string with the variable names 
of the parents. 

The conditional probability table (**cpt**) is a dict of the form 
`{(v1, v2, ...): p, ...}`, representing the distribution 
**P(X=true | parent1=v1, parent2=v2, ...) = p**. 
Each key is a tuple of boolean values corresponding to the parents. 
The length and order of the values in the keys must match the supplied **parents**. 
In all cases, the probability of **X** being false is left implicit, 
since it follows from **P(X=true)**.

The example below, implementing the well-known Burglary-Alarm network, 
will make this clearer.

<img src="images/bayesnet.png">

The alarm node can be created as follows: 

In [3]:
alarm_node = BayesNode('Alarm', ['Burglary', 'Earthquake'], 
                       {(True, True): 0.95,(True, False): 0.94, (False, True): 0.29, (False, False): 0.001})

It is possible to avoid using a tuple when there is only a single parent. So an alternative format for the **cpt** is

In [4]:
john_node = BayesNode('JohnCalls', ['Alarm'], {True: 0.90, False: 0.05})
mary_node = BayesNode('MaryCalls', 'Alarm', {(True, ): 0.70, (False, ): 0.01}) # Using string for parents.
# Equivalant to john_node definition.

The general format used for the alarm node always holds. For nodes with no parents we can also use.

In [5]:
burglary_node = BayesNode('Burglary', '', 0.001)
earthquake_node = BayesNode('Earthquake', '', 0.002)

It is possible to use the node for lookup function using the **p** method. The method takes in two arguments **value** and **event**. Event must be a dict of the type {variable:values, ..} The value corresponds to the value of the variable we are interested in (False or True).The method returns the conditional probability **P(X=value | parents=parent_values)**, where parent_values are the values of parents in event. (event must assign each parent a value.)

In [6]:
john_node.p(False, {'Alarm': True, 'Burglary': True}) # P(JohnCalls=False | Alarm=True)

0.09999999999999998

With all the information about nodes present it is possible to construct a Bayes Network using **BayesNet**. The **BayesNet** class does not take in nodes as input but instead takes a list of **node_specs**. An entry in **node_specs** is a tuple of the parameters we use to construct a **BayesNode** namely **(X, parents, cpt)**. **node_specs** must be ordered with parents before children.

In [7]:
psource(BayesNet)

class BayesNet:
    """Bayesian network containing only boolean-variable nodes."""

    def __init__(self, node_specs=None):
        """Nodes must be ordered with parents before children."""
        self.nodes: list[BayesNode] = []
        self.variables = []
        node_specs = node_specs or []
        for node_spec in node_specs:
            self.add(node_spec)

    def add(self, node_spec):
        """Add a node to the net. Its parents must already be in the
        net, and its variable must not."""
        # Unpack node_spec into a BayesNode and add it to self.nodes.
        # Update self.variables and the children list of each parent.
        X, parents, cpt = node_spec
        if X in self.variables:
            raise ValueError(f"Variable '{X}' already exists in the network!")
        
        new_node = BayesNode(X, parents, cpt)
        self.nodes.append(new_node)

        self.variables.append(X)
        for parent in new_node.parents:
            parent_node = self.variabl

The constructor of **BayesNet** takes each item in **node_specs** and adds a 
**BayesNode** to its **nodes** object variable by calling the **add** method. 
**add** in turn adds a node to the net. Its parents must already be in the net, 
and its variable must not. Thus **add** allows us to grow a **BayesNet** given 
its parents are already present.

**burglary** is a global instance of **BayesNet** corresponding to the above example.

In [8]:
T, F = True, False

burglary = BayesNet([('Burglary', '', 0.001),
                     ('Earthquake', '', 0.002),
                     ('Alarm', 'Burglary Earthquake',
                      {(T, T): 0.95, (T, F): 0.94, (F, T): 0.29, (F, F): 0.001}),
                     ('JohnCalls', 'Alarm', {T: 0.90, F: 0.05}),
                     ('MaryCalls', 'Alarm', {T: 0.70, F: 0.01})])

In [9]:
burglary

BayesNet([('Burglary', ''), ('Earthquake', ''), ('Alarm', 'Burglary Earthquake'), ('JohnCalls', 'Alarm'), ('MaryCalls', 'Alarm')])

**BayesNet** method **variable_node** allows to reach **BayesNode** instances inside a Bayes Net. It is possible to modify the **cpt** of the nodes directly using this method.

In [10]:
type(burglary.variable_node('Alarm'))

BayesianNetworks.BayesNode

In [11]:
burglary.variable_node('Alarm').cpt

{(True, True): 0.95,
 (True, False): 0.94,
 (False, True): 0.29,
 (False, False): 0.001}

## Exact Inference in Bayesian Networks

A Bayesian network is a compact representation of the full joint distribution. 
Like the full joint distribution, it allows us to perform inference — answering 
probabilistic queries about variables given some evidence.

Exact inference algorithms can answer such queries precisely, but they do not 
scale well for larger networks. Approximate methods, which trade accuracy for speed, 
are covered in the next section.

### Inference by Enumeration

We use techniques similar to those introduced for the full joint distribution 
to perform inference in Bayesian networks. The functions **enumeration_ask** 
and **enumerate_all** implement exact inference by enumerating all possibilities 
and summing them out.

In [12]:
psource(enumerate_all)

def enumerate_all(variables, e, bn: BayesNet):
    """Return the sum of those entries in P(variables | e{others})
    consistent with e, where P is the joint distribution represented
    by bn, and e{others} means e restricted to bn's other variables
    (the ones other than variables). Parents must precede children in variables."""
    if not variables:
        return 1.0
    Y, rest = variables[0], variables[1:]
    Ynode = bn.variable_node(Y)
    if Y in e:
        # Y is an evidence variable. Return its probability multiplied by the rest.
        return Ynode.p(e[Y], e) * enumerate_all(rest, e, bn)
    else:
        # Y is a hidden variable. Sum over all its possible values.
        return sum(Ynode.p(y, e) * enumerate_all(rest, extend(e, Y, y), bn)
                   for y in bn.variable_values(Y))



**enumerate_all** recursively evaluates a general form of the following equation:

$\textbf{P}(X | \textbf{e}) = \alpha \textbf{P}(X, \textbf{e}) = \alpha \sum_{y} \textbf{P}(X, \textbf{e}, \textbf{y})$ 

Here, **P(X, e, y)** is expressed as a product of conditional probabilities 
**P(variable | parents(variable))** obtained from the Bayesian network.

**enumeration_ask** calls **enumerate_all** for each possible value of the query 
variable **X** and normalizes the results to obtain the final probability distribution.

In [13]:
psource(enumeration_ask)

def enumeration_ask(X, e, bn: BayesNet):
    """Return the conditional probability distribution of variable X
    given evidence e, from BayesNet bn.
    >>> enumeration_ask('Burglary', dict(JohnCalls=T, MaryCalls=T), burglary
    ...  ).show_approx()
    'False: 0.716, True: 0.284'"""
    assert X not in e, "Query variable must be distinct from evidence"
    Q = ProbDist(X)
    for xi in bn.variable_values(X):
        # Compute P(X=xi | e) using enumerate_all.
        # Hint: Extend e with X=xi and pass all variables of bn.
        Q[xi] = enumerate_all(bn.variables, extend(e, X, xi), bn)
    return Q.normalize()



Let us solve the problem of finding out **P(Burglary=True | JohnCalls=True, MaryCalls=True)** using the **burglary** network. **enumeration_ask** takes three arguments **X** = variable name, **e** = Evidence (in form a dict like previously explained), **bn** = The Bayes Net to do inference on.

In [14]:
ans_dist = enumeration_ask('Burglary', {'JohnCalls': True, 'MaryCalls': True}, burglary)
ans_dist[True]

0.2841718353643929

### Variable Elimination

The enumeration algorithm can be improved substantially by eliminating repeated calculations. In enumeration we join the joint of all hidden variables. This is of exponential size for the number of hidden variables. Variable elimination employes interleaving join and marginalization.

Before we look into the implementation of Variable Elimination we must first familiarize ourselves with Factors. 

In general we call a multidimensional array of type P(Y1 ... Yn | X1 ... Xm) a factor where some of Xs and Ys maybe assigned values. Factors are implemented in the probability module as the class **Factor**. They take as input **variables** and **cpt**. 


#### Helper Functions

There are certain helper functions that help creating the **cpt** for the Factor given the evidence. Let us explore them one by one.

In [15]:
psource(make_factor)

def make_factor(var, e, bn: BayesNet):
    """Return the factor for var in bn's joint distribution given e."""
    node = bn.variable_node(var)
    # Collect variables (var + parents) that are not in evidence.
    variables = [X for X in [var] + node.parents if X not in e]
    # Build cpt by iterating over all events for these variables.
    cpt = {event_values(e1, variables): node.p(e1[var], e1)
           for e1 in all_events(variables, bn, e)}
    return Factor(variables, cpt)



**make_factor** is used to create the **cpt** and **variables** that will be passed to the constructor of **Factor**. We use **make_factor** for each variable. It takes in the arguments **var** the particular variable, **e** the evidence we want to do inference on, **bn** the bayes network.

Here **variables** for each node refers to a list consisting of the variable itself and the parents minus any variables that are part of the evidence. This is created by finding the **node.parents** and filtering out those that are not part of the evidence.

The **cpt** created is the one similar to the original **cpt** of the node with only rows that agree with the evidence.

In [16]:
psource(all_events)

def all_events(variables, bn: BayesNet, e):
    """Yield every way of extending e with values for all variables."""
    if not variables:
        yield e
    else:
        X, rest = variables[0], variables[1:]
        # Recursively extend e with each possible value of X.
        for e1 in all_events(rest, bn, e):
            for x in bn.variable_values(X):
                yield extend(e1, X, x)



The **all_events** function is a recursive generator that yields keys matching 
the original **cpt** of a node. It works by extending the evidence related to 
that node, so all generated events are consistent with the given evidence. 
Being a generator, it returns one such event per call.

We can demonstrate this with an example. Let us compute **f**<sub>5</sub>(A) = P(m | A).

In [17]:
f5 = make_factor('MaryCalls', {'JohnCalls': True, 'MaryCalls': True}, burglary)

In [18]:
f5

In [19]:
f5.cpt

{(True,): 0.7, (False,): 0.01}

In [20]:
f5.variables

['Alarm']

Here **f5.cpt** False key gives probability for **P(MaryCalls=True | Alarm = False)**. Due to our representation where we only store probabilities for only in cases where the node variable is True this is the same as the **cpt** of the BayesNode. Let us try a somewhat different example from the book where evidence is that the Alarm = True

In [21]:
new_factor = make_factor('MaryCalls', {'Alarm': True}, burglary)

In [22]:
new_factor.cpt

{(True,): 0.7, (False,): 0.30000000000000004}

Here the **cpt** is for **P(MaryCalls | Alarm = True)**. Therefore the probabilities for True and False sum up to one. Note the difference between both the cases. Again the only rows included are those consistent with the evidence.

#### Operations on Factors

We are interested in two kinds of operations on factors. **Pointwise Product** which is used to created joint distributions and **Summing Out** which is used for marginalization.

In [23]:
psource(Factor.pointwise_product)

    def pointwise_product(self, other, bn: BayesNet):
        """Multiply two factors, combining their variables."""
        # Compute union of variables from both factors.
        variables = list(set(self.variables) | set(other.variables))
        # Build cpt by multiplying probabilities for each event.
        cpt = {event_values(e, variables): self.p(e) * other.p(e)  for e in all_events(variables, bn, {})}
        return Factor(variables, cpt)



**Factor.pointwise_product** implements a method of creating a joint via combining two factors. We take the union of **variables** of both the factors and then generate the **cpt** for the new factor using **all_events** function. Note that the given we have eliminated rows that are not consistent with the evidence. Pointwise product assigns new probabilities by multiplying rows similar to that in a database join.

In [24]:
psource(pointwise_product)

def pointwise_product(factors, bn: BayesNet):
    return reduce(lambda f, g: f.pointwise_product(g, bn), factors)



**pointwise_product** extends this operation to more than two operands where it is done sequentially in pairs of two.

In [25]:
psource(Factor.sum_out)

    def sum_out(self, var, bn: BayesNet):
        """Make a factor eliminating var by summing over its values."""
        # Remove var from the variable list.
        variables = [X for X in self.variables if X != var]
        # Sum probabilities over all values of var.
        cpt = {event_values(e, variables): sum(self.p(extend(e, var, val))
               for val in bn.variable_values(var))
               for e in all_events(variables, bn, {})}
        return Factor(variables, cpt)



**Factor.sum_out** makes a factor eliminating a variable by summing over its values. Again **events_all** is used to generate combinations for the rest of the variables.

In [26]:
psource(sum_out)

def sum_out(var, factors, bn: BayesNet):
    """Eliminate var from all factors by summing over its values."""
    result, var_factors = [], []
    for f in factors:
        # Separate factors that contain var from those that don't.
        (var_factors if var in f.variables else result).append(f)
    # Multiply factors containing var, then sum out var.
    result.append(pointwise_product(var_factors, bn).sum_out(var, bn))
    return result



**sum_out** uses both **Factor.sum_out** and **pointwise_product** to finally eliminate a particular variable from all factors by summing over its values.

#### Elimination Ask

The function **elimination_ask** implements inference by variable elimination. 
The key idea is to eliminate hidden variables by interleaving joining and 
marginalization. It takes three arguments: **X** (the query variable), 
**e** (the evidence), and **bn** (the Bayesian network).

The algorithm creates factors from the Bayes nodes in reverse order and 
eliminates hidden variables using **sum_out**. Finally, it takes the 
pointwise product of all remaining factors and normalizes the result.

Let us solve the problem of inferring 
**P(Burglary=True | JohnCalls=True, MaryCalls=True)** using variable elimination.

In [27]:
psource(elimination_ask)

def elimination_ask(X, e, bn: BayesNet):
    """Compute bn's P(X|e) by variable elimination.
    >>> elimination_ask('Burglary', dict(JohnCalls=T, MaryCalls=T), burglary
    ...  ).show_approx()
    'False: 0.716, True: 0.284'"""
    assert X not in e, "Query variable must be distinct from evidence"
    factors = []
    for var in reversed(bn.variables):
        # Create a factor for this variable, then eliminate it if hidden.
        factors.append(make_factor(var, e, bn))
        if is_hidden(var, X, e):
            factors = sum_out(var, factors, bn)
    return pointwise_product(factors, bn).normalize()



In [28]:
elimination_ask('Burglary', dict(JohnCalls=True, MaryCalls=True), burglary).show_approx()

'False: 0.716, True: 0.284'

#### Elimination Ask: Optimizations and Considerations

`elimination_ask` has some critical points to consider, and several optimizations 
can be performed:

- **Operation on factors**:

  The `sum_out` and `pointwise_product` functions used in `elimination_ask` are 
  where the space and time complexity of the variable elimination algorithm arise.

  > The key insight is that any factor that does not depend on the variable being 
  > summed out can be moved outside the summation.

- **Variable ordering**:

  Elimination ordering is important. Every choice of ordering yields a valid 
  algorithm, but different orderings cause different intermediate factors to be 
  generated during the calculation. In this implementation, the algorithm applies 
  a reversed order.

  > In general, the time and space requirements of variable elimination are 
  > dominated by the size of the largest factor constructed during the operation 
  > of the algorithm. This in turn is determined by the order of elimination of 
  > variables and by the structure of the network. It turns out to be intractable 
  > to determine the optimal ordering, but several good heuristics are available. 
  > One fairly effective method is a greedy one: eliminate whichever variable 
  > minimizes the size of the next factor to be constructed.

- **Variable relevance**:
  
  Some variables may be irrelevant to resolving a query (i.e., their contribution 
  sums to 1). A variable elimination algorithm can therefore remove all such 
  variables before evaluating the query.

  > An optimization is to remove every variable that is not an ancestor of a query 
  > variable or evidence variable, as it is irrelevant to the query.

#### Runtime comparison
Let's see how the runtimes of these two algorithms compare.
We expect variable elimination to outperform enumeration by a large margin as we reduce the number of repetitive calculations significantly.

In [29]:
%%timeit
enumeration_ask('Burglary', dict(JohnCalls=True, MaryCalls=True), burglary).show_approx()

49.5 μs ± 3.16 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [30]:
%%timeit
elimination_ask('Burglary', dict(JohnCalls=True, MaryCalls=True), burglary).show_approx()

109 μs ± 24 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In this test case we observe that variable elimination is slower than what we expected. It has something to do with number of threads, how Python tries to optimize things and  this happens because the network is very small, with just 5 nodes. The `elimination_ask` has some critical point and some optimizations must be perfomed as seen above.
<br>
Of course, for more complicated networks, variable elimination will be significantly faster and runtime will drop not just by a constant factor, but by a polynomial factor proportional to the number of nodes, due to the reduction in repeated calculations.

## Approximate Inference in Bayesian Networks

Exact inference fails to scale for very large and complex Bayesian Networks. This section covers implementation of randomized sampling algorithms, also called Monte Carlo algorithms.

In [31]:
psource(BayesNode.sample)

    def sample(self, event: dict):
        """Sample from the distribution for this variable conditioned
        on event's values for parent_variables. That is, return True/False
        at random according with the conditional probability given the
        parents."""
        # Return True with probability P(self.variable=True | event).
        # Use self.p() and the probability() helper.
        p = self.p(True, event)
        return probability(p)



Before we consider the different algorithms in this section let us look at the **BayesNode.sample** method. It samples from the distribution for this variable conditioned on event's values for parent_variables. That is, return True/False at random according to with the conditional probability given the parents. The **probability** function is a simple helper from **utils** module which returns True with the probability passed to it.

### Prior Sampling

The idea of Prior Sampling is to sample from the Bayesian Network in a topological order. We start at the top of the network and sample as per **P(X<sub>i</sub> | parents(X<sub>i</sub>)** i.e. the probability distribution from which the value is sampled is conditioned on the values already assigned to the variable's parents. This can be thought of as a simulation.

In [32]:
psource(prior_sample)

def prior_sample(bn: BayesNet):
    """Randomly sample from bn's full joint distribution.
    The result is a {variable: value} dict."""
    # Implement this function.
    # Iterate over bn.nodes in order. For each node, sample a value
    # given the values already assigned to its parents in event.
    sample = {}
    for node in bn.nodes:
        sample[node.variable] = node.sample(sample)
    
    return sample



The function **prior_sample** generates random samples from the full joint 
distribution of a Bayesian network. Nodes are sampled in topological order, 
with the values of already-sampled parents passed as evidence. We will use 
the network shown below to demonstrate **prior_sample**.

<img src="images/sprinklernet.jpg" height="500" width="500">

Traversing the graph in topological order is essential. 
For this particular directed acyclic graph, there are two valid topological orderings:
<br>
1. `Cloudy -> Sprinkler -> Rain -> WetGrass`
2. `Cloudy -> Rain -> Sprinkler -> WetGrass`
<br>
<br>
We can follow either ordering to sample from the network. 
Any other ordering, however, is invalid.
<br>
One way to think about this is that `Cloudy` acts as a precondition for both 
`Rain` and `Sprinkler` — much like in planning, where preconditions must be 
satisfied before an action can be executed.
<br>
We store the generated samples as observations. Let us estimate 
**P(Rain=True)** by drawing 1000 random samples from the network.

In [33]:
sprinkler = BayesNet([('Cloudy', '', 0.5),
                      ('Sprinkler', 'Cloudy', {T: 0.10, F: 0.50}),
                      ('Rain', 'Cloudy', {T: 0.80, F: 0.20}),
                      ('WetGrass', 'Sprinkler Rain',
                       {(T, T): 0.99, (T, F): 0.90, (F, T): 0.90, (F, F): 0.00})])

In [34]:
N = 1000
all_observations = [prior_sample(sprinkler) for x in range(N)]

Now we filter to get the observations where Rain = True

In [35]:
rain_true = [observation for observation in all_observations if observation['Rain'] == True]

Finally, we can find **P(Rain=True)**

In [36]:
answer = len(rain_true) / N
print(answer)

0.496


Sampling this another time might give different results as we have no control over the distribution of the random samples

In [37]:
N = 1000
all_observations = [prior_sample(sprinkler) for x in range(N)]
rain_true = [observation for observation in all_observations if observation['Rain'] == True]
answer = len(rain_true) / N
print(answer)

0.529


To evaluate a conditional distribution. We can use a two-step filtering process. We first separate out the variables that are consistent with the evidence. Then for each value of query variable, we can find probabilities. For example to find **P(Cloudy=True | Rain=True)**. We have already filtered out the values consistent with our evidence in **rain_true**. Now we apply a second filtering step on **rain_true** to find **P(Rain=True and Cloudy=True)**

### Rejection Sampling

Rejection sampling is based on a simple idea. First, it generates samples from 
the prior distribution specified by the network. Then, it rejects all samples 
that do not match the evidence.

Rejection sampling is particularly useful when we know the query beforehand. 
While prior sampling works for any query in principle, it can fail in certain 
scenarios.

Consider a Bayesian network where we have evidence `e`, and we want to estimate 
how often state `A` is true given that evidence `e` is true. 
Normally, prior sampling can answer this question. However, if the probability 
of evidence `e` being true is very small in the actual distribution, it is 
possible that our random samples never contain a single instance where `e` is true. 
In that case, **P(e) = 0** in our sample, and **P(A | e) = P(A, e) / P(e)** 
becomes **0/0**, which is undefined. We cannot answer the query using this sample.

We could increase the number of samples, but we can never guarantee that we 
will encounter a case where `e` is true (even if such cases exist in the true 
distribution). To absolutely guarantee it, we would need to examine every 
possible data point, which means losing the speed advantage of approximation 
and effectively computing the exact inference.

This is where rejection sampling becomes valuable. Since we already know the 
query, we can simply reject any sample that is inconsistent with the evidence 
variables (in this example, the only evidence variable is `e`). 
We only consider samples that satisfy **all** evidence variables. 
In this way, we ensure we have enough relevant data to answer queries involving 
that evidence.

The function **rejection_sampling** implements this algorithm.

In [38]:
psource(rejection_sampling)

def rejection_sampling(X, e, bn: BayesNet, N=10000):
    """Estimate the probability distribution of variable X given
    evidence e in BayesNet bn, using N samples.
    Raises a ZeroDivisionError if all the N samples are rejected.
    >>> random.seed(47)
    >>> rejection_sampling('Burglary', dict(JohnCalls=T, MaryCalls=T),
    ...   burglary, 10000).show_approx()
    'False: 0.7, True: 0.3'
    """
    # Implement rejection sampling.
    # 1. Generate N prior samples.
    # 2. Keep only those consistent with evidence e.
    # 3. Count how often X takes each value.
    # 4. Return a ProbDist over X.
    consistent_samples = []
    for _ in range(N):
        sample = prior_sample(bn)
        if consistent_with(sample, e):
            consistent_samples.append(sample)

    if len(consistent_samples) == 0:
        raise ZeroDivisionError("All the N samples are rejected!")   
    
    counts_X = {True: 0, False: 0}
    for sample in consistent_samples:
        counts_X[sample[X]] += 1
   

The function keeps counts of each of the possible values of the Query variable and increases the count when we see an observation consistent with the evidence. It takes in input parameters **X** - The Query Variable, **e** - evidence, **bn** - Bayes net and **N** - number of prior samples to generate.

**consistent_with** is used to check consistency.

In [39]:
psource(consistent_with)

def consistent_with(event, evidence):
    """Is event consistent with the given evidence?"""
    # Implement this function.
    # Return True iff for every var in evidence, event[var] == evidence[var].
    for var in evidence.keys():
        if event[var] != evidence[var]:
            return False
        
    return True



To answer **P(Cloudy=True | Rain=True)**

In [40]:
p = rejection_sampling('Cloudy', dict(Rain=True), sprinkler, 1000)
p[True]

0.7838899803536346

### Likelihood Weighting

Rejection sampling becomes inefficient when the probability of finding samples 
consistent with the evidence is low. It is also slow for larger networks or 
when the evidence consists of many variables, since most samples end up being 
rejected.

Likelihood weighting solves this problem by fixing the evidence variables 
(i.e., not sampling them) and instead assigning a weight to each sample to 
ensure the overall sampling remains consistent with the evidence.

The functions **likelihood_weighting** and **weighted_sample** implement 
this algorithm.

In [41]:
psource(weighted_sample)

def weighted_sample(bn: BayesNet, e):
    """Sample an event from bn that's consistent with the evidence e;
    return the event and its weight."""
    # Implement this function.
    # 1. Start with event = dict(e) and weight w = 1.
    # 2. For each node in topological order:
    #    - If node is in e: multiply w by P(node=evidence_value | event).
    #    - If node is not in e: sample its value from P(node | event).
    # 3. Return (event, w).
    event = dict(e)
    w = 1.0
    for node in bn.nodes:
        if node.variable in e:
            w *= node.p(e[node.variable], event)
        else:
            event[node.variable] = node.sample(event)

    return (event, w)




**weighted_sample** samples an event from Bayesian Network that's consistent with the evidence **e** and returns the event and its weight, the likelihood that the event accords to the evidence. It takes in two parameters **bn** the Bayesian Network and **e** the evidence.

The weight is obtained by multiplying **P(x<sub>i</sub> | parents(x<sub>i</sub>))** for each node in evidence. We set the values of **event = evidence** at the start of the function.

In [42]:
weighted_sample(sprinkler, dict(Rain=True))

({'Rain': True, 'Cloudy': True, 'Sprinkler': False, 'WetGrass': True}, 0.8)

In [43]:
psource(likelihood_weighting)

def likelihood_weighting(X, e, bn: BayesNet, N=10000):
    """Estimate the probability distribution of variable X given
    evidence e in BayesNet bn.
    >>> random.seed(1017)
    >>> likelihood_weighting('Burglary', dict(JohnCalls=T, MaryCalls=T),
    ...   burglary, 10000).show_approx()
    'False: 0.702, True: 0.298'
    """
    # Implement likelihood weighting.
    # 1. Generate N weighted samples (use weighted_sample).
    # 2. For each sample, accumulate its weight into the count for sample[X].
    # 3. Return a ProbDist over X.
    weights_X = {True: 0, False: 0}
    for _ in range(N):
        sample, w = weighted_sample(bn, e)
        weights_X[sample[X]] += w
    
    return ProbDist(X, weights_X)



**likelihood_weighting** implements the algorithm to solve our inference problem. The code is similar to **rejection_sampling** but instead of adding one for each sample we add the weight obtained from **weighted_sampling**.

In [44]:
likelihood_weighting('Cloudy', dict(Rain=True), sprinkler, 200).show_approx()

'False: 0.206, True: 0.794'

### Gibbs Sampling

In likelihood weighting, it is possible to obtain very low weights when the 
evidence variables are located near the leaves of the Bayesian network. This 
happens because, in likelihood weighting, influence only propagates downward 
— from parents to children.

Gibbs sampling solves this limitation. Instead of generating each sample from 
scratch, it starts from a random state consistent with the evidence and repeatedly 
updates one variable at a time by sampling from its distribution conditioned on 
the current values of all other variables (its Markov blanket). Over many iterations, 
this process converges to the true posterior distribution.

The function **gibbs_ask** implements this algorithm.

In [45]:
psource(gibbs_ask)

def gibbs_ask(X, e, bn: BayesNet, N=1000):
    """Estimate the probability distribution of variable X given
    evidence e in BayesNet bn using Gibbs sampling."""
    assert X not in e, "Query variable must be distinct from evidence"
    # Implement Gibbs sampling.
    # 1. Initialize state with evidence e and random values for other variables.
    # 2. For N iterations:
    #    - For each non-evidence variable Zi, resample it from markov_blanket_sample.
    #    - Increment the count for the current value of X.
    # 3. Return a ProbDist over X.
    state = dict(e)
    Z = [node.variable for node in bn.nodes if node.variable not in e]
    for var in Z:
        state[var] = random.choice([True, False])

    counts_X = {True: 0, False: 0}
    for _ in range(N):
        for Zi in Z:
            state[Zi] = markov_blanket_sample(Zi, state, bn)
            
        counts_X[state[X]] += 1
    
    return ProbDist(X, counts_X)



In **gibbs_ask** we initialize the non-evidence variables to random values. And then select non-evidence variables and sample it from **P(Variable | value in the current state of all remaining vars) ** repeatedly sample. In practice, we speed this up by using **markov_blanket_sample** instead. This works because terms not involving the variable get canceled in the calculation. The arguments for **gibbs_ask** are similar to **likelihood_weighting**

In [46]:
gibbs_ask('Cloudy', dict(Rain=True), sprinkler, 200).show_approx()

'False: 0.195, True: 0.805'

#### Runtime analysis
Let's take a look at how much time each algorithm takes.

In [47]:
%%timeit
all_observations = [prior_sample(sprinkler) for x in range(1000)]
rain_true = [observation for observation in all_observations if observation['Rain'] == True]
len([observation for observation in rain_true if observation['Cloudy'] == True]) / len(rain_true)

2.22 ms ± 84.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [48]:
%%timeit
rejection_sampling('Cloudy', dict(Rain=True), sprinkler, 1000)

2.42 ms ± 80.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [49]:
%%timeit
likelihood_weighting('Cloudy', dict(Rain=True), sprinkler, 200)

452 μs ± 9.29 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [51]:
%%timeit
gibbs_ask('Cloudy', dict(Rain=True), sprinkler, 200)

1.94 ms ± 77.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


As expected, all algorithms have a very similar runtime.
However, rejection sampling would be a lot faster and more accurate when the probabiliy of finding data-points consistent with the required evidence is small.
<br>
Likelihood weighting is the fastest out of all as it doesn't involve rejecting samples, but also has a quite high variance.